# Демо: хэширование

Каждый раз, когда мы кладём ключ в `dict` или элемент в `set`, Python под капотом вызывает `hash(obj)` — берёт «отпечаток» объекта в виде числа. На этом отпечатке держится весь O(1) доступ. Прокликаем Shift+Enter и посмотрим, у каких объектов этот отпечаток есть, у каких нет, и что с этим делать.

## Часть 1. Функция `hash()`

Сейчас посмотрим, что возвращает `hash()` для типичных hashable объектов: числа, строки, кортежи. Главное наблюдение: одинаковые объекты дают одинаковый хэш, разные — почти всегда разный.

In [1]:
# Числа: hash целого числа равен ему самому (для небольших int)
print(hash(1))               # 1
print(hash(42))              # 42
print(hash(-7))              # -7
print(hash(3.14))            # большое число — float хэшируется иначе

1
42
-7
322818021289917443


In [2]:
# Строки: одинаковая строка → одинаковый хэш в текущей сессии
h1 = hash('привет')
h2 = hash('привет')
h3 = hash('пока')
print(h1)                    # большое число (зависит от запуска Python)
print(h1 == h2)              # True — одинаковые строки
print(h1 == h3)              # False — разные строки

5249656416355818969
True
False


In [3]:
# Кортежи: хэшируются, если все элементы хэшируются
print(hash((1, 2)))          # ок, int хэшируются
print(hash(('a', 'b', 'c'))) # ок, str хэшируются
print(hash((1, 'two', 3.0))) # ок, смешанные hashable элементы

# Контракт: равные объекты — равные хэши
print(hash((1, 2)) == hash((1, 2)))  # True

-3550055125485641917
2630115593417936820
2558980110379075894
True


## Часть 2. Unhashable типы

А вот что произойдёт, если попробовать вычислить `hash()` от изменяемого объекта — списка, словаря или множества. Python кинет `TypeError`. Оборачиваем вызовы в `try/except`, чтобы увидеть текст ошибки и продолжить выполнение ноутбука.

In [4]:
# Список — unhashable
try:
    hash([1, 2, 3])
except TypeError as e:
    print(f'TypeError: {e}')   # unhashable type: 'list'

TypeError: unhashable type: 'list'


In [5]:
# Словарь — тоже unhashable
try:
    hash({'name': 'Аня'})
except TypeError as e:
    print(f'TypeError: {e}')   # unhashable type: 'dict'

TypeError: unhashable type: 'dict'


In [6]:
# Множество — unhashable (но frozenset — hashable, см. ниже)
try:
    hash({1, 2, 3})
except TypeError as e:
    print(f'TypeError: {e}')   # unhashable type: 'set'

# frozenset — неизменяемая версия set, её хэшировать можно
print(hash(frozenset({1, 2, 3})))  # большое число — ок

TypeError: unhashable type: 'set'
-272375401224217160


Следствие: такие объекты нельзя класть как ключи `dict` или элементы `set`. Python проверяет hashable на этапе вставки и падает с тем же `TypeError`.

In [7]:
# Список как ключ dict — TypeError
try:
    d = {[1, 2]: 'значение'}
except TypeError as e:
    print(f'TypeError: {e}')   # unhashable type: 'list'

# Список как элемент set — тоже TypeError
try:
    s = {[1, 2], [3, 4]}
except TypeError as e:
    print(f'TypeError: {e}')   # unhashable type: 'list'

TypeError: unhashable type: 'list'
TypeError: unhashable type: 'list'


## Часть 3. Почему изменяемые объекты — unhashable

Запрет неслучайный. Представим, что Python всё-таки разрешил бы класть список как ключ. Произошло бы вот что:

1. Кладём `key = [1, 2]` в словарь. Python вычислил `hash([1, 2])` — пусть это число `42`. Пара лежит в ячейке `42 % размер_таблицы`.
2. Мутируем тот же список: `key.append(3)`. Теперь это `[1, 2, 3]` и его хэш — другой, например `99`. Указывает на другую ячейку.
3. Ищем `d[key]`. Python считает хэш текущего значения списка — `99`, идёт в ячейку `99 % размер_таблицы`, не находит ничего. **Данные потеряны** — словарь не может их найти, хотя они физически лежат в ячейке `42`.

Демонстрация на кортеже (он immutable — таких проблем нет).

In [8]:
# Кортеж как ключ — работает корректно, потому что immutable
distances = {}
key = (1, 2)
distances[key] = 'origin → (1,2)'
print(distances)

# Хэш кортежа стабилен — один и тот же при каждом обращении
print(hash(key))             # одно число
print(hash((1, 2)))          # то же число — одинаковые кортежи

{(1, 2): 'origin → (1,2)'}
-3550055125485641917
-3550055125485641917


Если очень нужен «список» как ключ — конвертируй его в `tuple`. Если нужно множество в качестве ключа — используй `frozenset`.

In [9]:
# Конвертация изменяемого в неизменяемое перед использованием как ключ
raw = [1, 2, 3]
d = {tuple(raw): 'значение'}    # list → tuple
print(d)                        # {(1, 2, 3): 'значение'}

raw_set = {1, 2, 3}
d2 = {frozenset(raw_set): 'значение'}  # set → frozenset
print(d2)                       # {frozenset({1, 2, 3}): 'значение'}

{(1, 2, 3): 'значение'}
{frozenset({1, 2, 3}): 'значение'}


## Часть 4. А что со своими типами?

Когда дойдём до классов, узнаешь: свой пользовательский тип можно сделать хэшируемым через два магических метода — `__eq__` (когда объекты равны) и `__hash__` (как считать отпечаток). Без них два объекта с одинаковым содержимым считаются разными ключами словаря (сравнение идёт по `id()` — адресу в памяти).

Это разберём подробно на третьей неделе, в теме «Магические методы». Сейчас достаточно знать: встроенные immutable типы (`int`, `str`, `tuple`, `frozenset`) хэшируются «из коробки», а для своих классов придётся писать `__eq__` + `__hash__` руками — или использовать `@dataclass(frozen=True)`, который сгенерирует их автоматически.

## Мини-задания

Три упражнения. Подсказок к методам и функциям нет — вспомни сам.

**Задание 1.** Дан список чисел `nums = [1, 2, 2, 3, 1, 4, 5, 3]`. Получи только уникальные значения. Ожидаемый результат — коллекция, содержащая `1, 2, 3, 4, 5` (в любом порядке).

**Задание 2.** Дан список слов `words = ['яблоко', 'банан', 'яблоко', 'вишня', 'банан', 'яблоко']`. Посчитай, сколько раз встречается каждое слово. Ожидаемый результат: `{'яблоко': 3, 'банан': 2, 'вишня': 1}`.

**Задание 3.** Даны две географические координаты в виде списков: `coord_a = [55.75, 37.62]` (Москва) и `coord_b = [59.93, 30.31]` (Питер). Сделай так, чтобы их можно было использовать как ключи словаря с названиями городов. Ожидаемый результат — словарь, в котором по координате можно достать имя города.

In [10]:
# Задание 1
nums = [1, 2, 2, 3, 1, 4, 5, 3]
# Твой код:

# Ожидаемый результат содержит {1, 2, 3, 4, 5}

In [11]:
# Задание 2
words = ['яблоко', 'банан', 'яблоко', 'вишня', 'банан', 'яблоко']
counts = {}
# Твой код:

# Ожидаемый counts: {'яблоко': 3, 'банан': 2, 'вишня': 1}

In [12]:
# Задание 3
coord_a = [55.75, 37.62]
coord_b = [59.93, 30.31]
cities = {}
# Твой код — преврати координаты во что-то hashable и положи в cities

# Ожидаемый результат: словарь из 2 элементов, где ключ — координата,
# значение — название города ('Москва' / 'Санкт-Петербург')